In [ ]:
# Jupyter Notebook - 代码
# 导入必要的库
import matplotlib.pyplot as plt
import numpy as np
import os,shutil
import tensorflow as tf
import seaborn as sns
from tqdm import tqdm
import datetime
import tensorflow_model_optimization as tfmot
from lib import AU
# 设定日志级别
tf.get_logger().setLevel('ERROR')

# 🔹 超参数
IMG_SIZE = (160, 160)
AUTOTUNE = tf.data.AUTOTUNE
IMG_SHAPE = IMG_SIZE + (3,)

# 创建model目录（如果不存在）
model_dir = 'model'
os.makedirs(model_dir, exist_ok=True)

阶段一

In [ ]:
# 🔹 准备数据集
shutil.rmtree('cache', ignore_errors=True)
os.makedirs('cache', exist_ok=True)

BATCH_SIZE = 16
base_dir = '../Datasets/smartcar26_dataset' 
train_dir = os.path.join(base_dir, 'train')
valid_dir = os.path.join(base_dir, 'val')

train_raw = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, batch_size=BATCH_SIZE, image_size=IMG_SIZE)

validation_raw = tf.keras.preprocessing.image_dataset_from_directory(
    valid_dir, batch_size=BATCH_SIZE, image_size=IMG_SIZE)

class_names = train_raw.class_names
print("Class Names:", class_names)

# 获取时间戳并创建唯一的文件夹，后续所有输出都存放在这里
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
model_folder_path = f"./model/model_{timestamp}"
os.makedirs(model_folder_path, exist_ok=True)

# 阶段 1 数据准备 (不带增强)
train_dataset = (train_raw
                 .map(AU.preprocess_image, num_parallel_calls=AUTOTUNE)
                 .cache(os.path.join('cache', 'train_cache1'))
                 .shuffle(1000)
                 .prefetch(AUTOTUNE))

validation_dataset = (validation_raw
                      .map(AU.preprocess_image, num_parallel_calls=AUTOTUNE)
                      .cache(os.path.join('cache', 'val_cache'))
                      .prefetch(AUTOTUNE))

# 预热缓存
print("开始预热缓存...")
for _ in tqdm(train_dataset, desc="Caching train"): pass
for _ in tqdm(validation_dataset, desc="Caching val"): pass

In [ ]:
# 🔹 构建模型

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE, 
    include_top=False, 
    pooling = 'avg', 
    alpha=0.35, 
    # include_preprocessing=False,
    weights='imagenet')

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),
    base_model,
    tf.keras.layers.Dropout(0.8),
    tf.keras.layers.Dense(len(class_names),activation='softmax')
])
model.build((None, 160, 160, 3))
model.summary()

In [ ]:
# 编译并训练阶段 1
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1.5e-5, decay_steps=len(train_dataset), decay_rate=0.99, staircase=True)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

history1 = model.fit(train_dataset, validation_data=validation_dataset, epochs=30, callbacks=[early_stopping])

# 直接保存到目标文件夹
stage1_path = os.path.join(model_folder_path, 'stage1_model.h5')
model.save(stage1_path)
print(f"Stage 1 model saved to: {stage1_path}")

阶段二

In [ ]:
# 阶段 2 数据准备 (带增强)
# 复用 validation_dataset 和 validation_raw
train_dataset2 = (train_raw
                 .map(AU.preprocess_image_aug, num_parallel_calls=AUTOTUNE)
                 .cache(os.path.join('cache', 'train_cache2'))
                 .shuffle(1000)
                 .prefetch(AUTOTUNE))

print("开始预热阶段 2 缓存...")
for _ in tqdm(train_dataset2, desc="Caching train stage2"): pass

In [ ]:
# 训练阶段 2
# model 已经在内存中且处于阶段 1 后的状态，直接继续训练
history2 = model.fit(train_dataset2, validation_data=validation_dataset, epochs=10, 
                    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

# 直接保存到目标文件夹
stage2_path = os.path.join(model_folder_path, 'stage2_model.h5')
model.save(stage2_path)
print(f"Stage 2 model saved to: {stage2_path}")

In [ ]:
# 🔹 导出 TFLite
def representative_dataset():
    calibration_ds = (validation_raw.map(AU.preprocess_image, num_parallel_calls=AUTOTUNE)
                      .take(500).cache().prefetch(AUTOTUNE))
    for images, _ in tqdm(calibration_ds, desc="Calibration"):
        yield [tf.cast(images, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

# 保存 TFLite 和标签到已创建的文件夹 (固定名称为 model.tflite)
tflite_path = os.path.join(model_folder_path, "model.tflite")
with open(tflite_path, 'wb') as f: f.write(tflite_model)

with open(os.path.join(model_folder_path, "labels.txt"), "w", encoding="utf-8") as f:
    for name in class_names: f.write(f"{name}\n")

print(f"TFLite 模型已保存至: {tflite_path}")

In [ ]:
from sklearn.metrics import confusion_matrix
# 混淆矩阵
y_pred = np.argmax(model.predict(validation_dataset), axis=1)
y_true = np.concatenate([labels.numpy() for _, labels in validation_dataset])

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap="Blues", fmt="d", 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

# 保存混淆矩阵到模型文件夹
if 'model_folder_path' in locals():
    cm_path = os.path.join(model_folder_path, "confusion_matrix.png")
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    print(f"混淆矩阵已保存至: {cm_path}")

plt.show()

In [ ]:
from lib import polt_improved

# 绘制并保存曲线
polt_improved.plot_combined_curves_improved([history1, history2], save_dir=model_folder_path)

# 清理缓存文件夹
shutil.rmtree('cache', ignore_errors=True)
print("Cleared cache directory.")

In [ ]:
# 🔹 联调测试
import model_test

test_dir = os.path.join(base_dir, 'test')
if not os.path.exists(test_dir):
    test_dir = os.path.join(os.path.dirname(base_dir), 'smartcar26_dataset', 'test')

if os.path.exists(test_dir) and os.path.exists(tflite_path):
    print(f"Testing model: {tflite_path}")
    model_test.main(model_path=tflite_path, test_dir=test_dir)
else:
    print("Test directory or TFLite model not found.")